In [2]:
# Connecting to huggin face and downloading dataset 
import os
from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import login

# Checking if it is running on Google Colab or locally
try:
    from google.colab import userdata
    import gdown

    hf_token = userdata.get('HF_TOKEN')

    url = 'https://drive.google.com/file/d/18f60tbAjVKYgfqu2ILngBw2wLjwF5lv-/view?usp=sharing'
    gdown.download(url, quiet=True, fuzzy=True)

    dataset = "./corpora.parquet"

except ModuleNotFoundError as e:
    hf_token = os.getenv("HF_TOKEN")
    dataset = "../../corpora/GeneratedCorpus/corpora.parquet"

login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
import gc
import torch
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import json
import logging
from tqdm import tqdm
from glob import glob
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# pip install bitsandbytes accelerate


logging.basicConfig(
    filename="experiment_errors.log",
    level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8",
)

In [5]:
def clean_memory():
    """Force RAM and VRAM to clean"""
    gc.collect()                # Clean not used python objects
    torch.cuda.empty_cache()    # Clean Pytorch alocation cache
    print("Memory cleaned 🧹!")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Execution environment: {device.upper()}")

Execution environment: CUDA


In [6]:
df = pd.read_parquet(dataset)

SCENARIOS = df["scenario"].unique().tolist()
print(SCENARIOS)

LANGAUGES = df["language"].unique().tolist()
print(LANGAUGES)

TASKS = df["task"].unique().tolist()
print(TASKS)

TASK_TYPES = df["task_type"].unique().tolist()
print(TASK_TYPES)

df.tail()

['GRAMMAR SWAP', 'LEXICAL SWAP', 'TRANSLATION', 'PB NATIVO', 'PB PERMUTED', 'PB CORRUPTED']
['INGLÊS', 'MANDARIM', 'ÁRABE', 'ESPANHOL', 'FRANCÊS', 'GALEGO', 'RUSSO', 'PB NATIVO']
['CAPACITAR', 'EXPLICAR', 'COMPARTILHAR', 'FAZER', 'RECRIAR', 'RELATAR', 'EXPLORAR', 'RECOMENDAR']
['ED', 'FD']


,scenario,language,task,task_type,text
1511,PB CORRUPTED,PB NATIVO,RECRIAR,ED,"A loura \n\n\n- Pai, Contados aquela história ..."
1512,PB CORRUPTED,PB NATIVO,RECRIAR,FD,ININTELIGÍVEL \nFALANTE1 hi hello hello tudo b...
1513,PB CORRUPTED,PB NATIVO,RECRIAR,FD,ININTELIGÍVEL \n\nFALANTE1 center center Adora...
1514,PB CORRUPTED,PB NATIVO,RECRIAR,FD,JOÃO boa noite \nPAULO boa noite \nJOÃO meu no...
1515,PB CORRUPTED,PB NATIVO,RECRIAR,FD,FALANTE1 hoje eu vou embrulhar esse santo antô...


In [14]:
def load_model_and_tokenizer(model_url:str, quantitize = False):
    tokenizer = AutoTokenizer.from_pretrained(
        model_url,
        use_fast=False,
        trust_remote_code=True
    )

    bnb_config = None
    if quantitize:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_url,
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map="auto",
        dtype=torch.float16
    )

    # model.to(device)
    model.eval()

    return tokenizer, model

In [ ]:
def eval_causal_sliding(model:AutoModelForCausalLM, tokenizer:AutoTokenizer, corpus:pd.DataFrame, stride=512, max_len=None):
    results = {
        "text_id": [],
        "num_words": [], 
        "num_bytes": [], 
        "num_tokens": [],
        "tokens_per_word": [],
        "num_loss_tokens": [],
        "average_nll": [],
        "total_nll": [],
        "ppl": [],
        "bpb": [],
    }

    if max_len is None:
        try:
            max_len = model.config.max_position_embeddings
        except AttributeError:
            max_len = model.config.text_config.max_position_embeddings

    if stride <= 0:
        raise ValueError("stride must be positive")
    if stride > max_len:
        raise ValueError("stride cannot be greater than max_len")

    for text_id, row in tqdm(corpus.iterrows(), desc="Processing corpus", unit="texts"):
        text = row["text"]

        num_bytes = len(text.encode("utf-8"))
        num_words = len(text.split())
                
        inputs = tokenizer(text, return_tensors="pt").to(device)
        num_tokens = inputs.input_ids.size(1)

        tokens_per_word = num_tokens / num_words if num_words > 0 else float("nan")

        total_nll = 0.0
        total_loss_tokens = 0
        prev_end_window = 0

        # Sliding window
        for begin_window in range(0, num_tokens, stride):
            end_window = min(begin_window + max_len, num_tokens)

            # Since the windows overlap, this indicates how many new tokens
            # will be actually used to calculate the perplexity
            target_len = end_window - prev_end_window

            # Select only the tokens from the window
            input_ids = inputs.input_ids[:, begin_window:end_window]
            target_ids = input_ids.clone()

            # Loss will ignore labels with value -100
            target_ids[:, :-target_len] = -100

            with torch.no_grad():
                outputs = model(input_ids, labels=target_ids)

            num_loss_tokens = (target_ids[:, 1:] != -100).sum().item()

            if num_loss_tokens > 0:
                total_nll += outputs.loss.item() * num_loss_tokens
                total_loss_tokens += num_loss_tokens
            
            prev_end_window = end_window

            if end_window == num_tokens:
                break


        if total_loss_tokens > 0:
            average_nll = total_nll / total_loss_tokens
            ppl = math.exp(average_nll)
        else:
            average_nll = float("nan")
            ppl = float("nan")

        if num_bytes > 0 and total_loss_tokens > 0:
            bpb = total_nll / (math.log(2) * num_bytes)
        else:
            bpb = float("nan")


        results["text_id"].append[text_id]
        results["num_words"].append[num_words]
        results["num_bytes"].append[num_bytes]
        results["num_tokens"].append[num_tokens]
        results["tokens_per_word"].append[tokens_per_word]
        results["average_nll"].append[average_nll]
        results["total_nll"].append[total_nll]
        results["ppl"].append[ppl]
        results["bpb"].append[bpb]
    
    return results

In [ ]:
def execute_experiment(model_name, model_url, quantization, corpus:pd.DataFrame, output_file, scenarios:list|str="all"):
    
    # preparing the selected corpus to evaluate
    dataset = corpus.copy()
    if scenarios != "all":
        dataset = dataset[dataset["scenario"].isin(scenarios)]

    try:
        print(f"⏳ Loading {model_name}")
        model, tokenizer = load_model_and_tokenizer(model_url)
    except Exception:
        logging.exception(
            "Error loading model %s from %s with quantization=%s",
            model_name,
            model_url,
            quantization,
        )
        return None
    

SyntaxError: incomplete input (4099893969.py, line 3)

In [ ]:
df["text"].tolist()[0]

'Fondant / Pasta de goma\n\nEi todos, boa noite!\nEu-sou Tati, eu-sou nova aqui e eu tenho uma questão:\neu amo decorações de fondant e eu-tenho sempre querido aprender como trabalhar com isso, mas eu apenas me-mudei para os EUA. Eu-tenho estado procurando por aulas mas eu não-posso descobrir o quê americanos chamam fondant. Eu-tenho procurado todo-acima e encontrado ao menos cinco tipos diferentes de produtos. Alguém sabe o quê isso é chamado aqui? Eu sei que matéria usada para decorar bolos geralmente vai por "icing", mas eu simplesmente não-posso encontrar fondant especificamente.\nQualquer ajuda seria ótima. Obrigada em antecipado, todos!\n\nRE: Fondant\n\nEi Tati, bem-vinda ao fórum!\nFondant realmente vai por alguns nomes diferentes aqui. Eu penso o mais comum um americanos usam é "rolled fondant". Há também "gum paste". Os Britânicos tendem a dizer "sugarcraft". Google qualquer um desses e tu-vais-encontrar toneladas de sites com informação, aulas, suprimentos — as obras.'